<a href="https://colab.research.google.com/github/thiranesh/Thiranesh-codeboosters-2026/blob/main/day-4/Day_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
!pip install pyspark --quiet
print('PySpark installation complete!')

PySpark installation complete!


In [6]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.functions import year, month, to_date, col, round as spark_round
import matplotlib.pyplot as plt
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

spark = SparkSession.builder \
  .appName('Day4_BigData_Sales') \
  .config('spark.sql.adaptive.enabled','true') \
  .getOrCreate()

print(f'Spark Version : {spark.version}')
print(f'SparkSession : ACTIVE')
print(f'Application : {spark.sparkContext.appName}')

Spark Version : 4.0.2
SparkSession : ACTIVE
Application : Day4_BigData_Sales


In [7]:
df_bronze = spark.read \
  .option('header','true') \
  .option('inferSchema', 'true') \
  .csv('large_sales_data.csv')

print('===Bronze Layer - Raw Data===')
print(f'Rows:{df_bronze.count()}')
print(f'Columns:{len(df_bronze.columns)}')
print(f'Names:{df_bronze.columns}')
df_bronze.printSchema()

===Bronze Layer - Raw Data===
Rows:5000
Columns:13
Names:['order_id', 'customer_name', 'product', 'category', 'quantity', 'unit_price', 'revenue', 'order_date', 'city', 'region', 'sales_rep', 'payment_method', 'order_status']
root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)



In [8]:
print('First 5 rows:')
df_bronze.show(5, truncate=False)

print('\nBasic statistics for numeric columns:')
df_bronze.select('quantity','unit_price','revenue').describe().show()

First 5 rows:
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|order_id|customer_name|product   |category   |quantity|unit_price|revenue|order_date|city     |region|sales_rep  |payment_method  |order_status|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|1001    |Sneha Reddy  |Monitor   |Electronics|12      |22000     |264000 |2023-05-21|Mumbai   |West  |Meera Patel|UPI             |Delivered   |
|1002    |Ramesh Kumar |Printer   |Electronics|10      |12000     |120000 |2023-08-05|Delhi    |North |Anil Sharma|Credit Card     |Shipped     |
|1003    |Rahul Mishra |Mouse     |Accessories|10      |800       |8000   |2023-01-14|Ahmedabad|West  |Meera Patel|Cash on Delivery|Shipped     |
|1004    |Suresh Rao   |Tablet    |Electronics|5       |32000     |160000 |2023-01-04|Surat    |West  |Ravi Ku

In [9]:
df_bronze.write \
  .mode('overwrite') \
  .parquet('sales_bronze.parquet')
print('Bronze Parquet saved: sales_bronze.parquet')

import os

def get_dir_size(path):
  if os.path.isfile(path):
    return os.path.getsize(path) /1024
  total = 0
  for dirpath, dirnames, filenames in os.walk(path):
    for f in filenames:
      total += os.path.getsize(os.path.join(dirpath, f))
  return total /1024

csv_size = get_dir_size('large_sales_data.csv')
parquet_size = get_dir_size('sales_bronze.parquet')
reduction = (1 - (parquet_size / csv_size)) * 100

print(f'CSV Size: {csv_size:.1f} KB')
print(f'Parquet Size: {parquet_size:.1f} KB')
print(f'Reduction: {reduction:.1f}%')
print(f'\nAt 1 TB  scale: csv=1000 GB -> Parquet= {1000 * (parquet_size / 100):.0f} GB')

Bronze Parquet saved: sales_bronze.parquet
CSV Size: 529.3 KB
Parquet Size: 55.1 KB
Reduction: 89.6%

At 1 TB  scale: csv=1000 GB -> Parquet= 551 GB


In [10]:
print('First 5 rows:')
df_bronze.show(5, truncate=False)

# Correctly add a new column 'unit_price_revenue_sum' that is the sum of 'unit_price' and 'revenue'
df_bronze = df_bronze.withColumn('unit_price_revenue_sum', F.col('unit_price') + F.col('revenue'))

print('\nFirst 5 rows with new sum column:')
df_bronze.show(5, truncate=False)

print('\nBasic statistics for numeric columns:')
df_bronze.select('quantity','unit_price','revenue').describe().show()

First 5 rows:
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|order_id|customer_name|product   |category   |quantity|unit_price|revenue|order_date|city     |region|sales_rep  |payment_method  |order_status|
+--------+-------------+----------+-----------+--------+----------+-------+----------+---------+------+-----------+----------------+------------+
|1001    |Sneha Reddy  |Monitor   |Electronics|12      |22000     |264000 |2023-05-21|Mumbai   |West  |Meera Patel|UPI             |Delivered   |
|1002    |Ramesh Kumar |Printer   |Electronics|10      |12000     |120000 |2023-08-05|Delhi    |North |Anil Sharma|Credit Card     |Shipped     |
|1003    |Rahul Mishra |Mouse     |Accessories|10      |800       |8000   |2023-01-14|Ahmedabad|West  |Meera Patel|Cash on Delivery|Shipped     |
|1004    |Suresh Rao   |Tablet    |Electronics|5       |32000     |160000 |2023-01-04|Surat    |West  |Ravi Ku

In [11]:
df_bronze.filter(F.col("revenue")>40000).show()

+--------+-------------+-------+-----------+--------+----------+-------+----------+---------+------+------------+----------------+------------+----------------------+
|order_id|customer_name|product|   category|quantity|unit_price|revenue|order_date|     city|region|   sales_rep|  payment_method|order_status|unit_price_revenue_sum|
+--------+-------------+-------+-----------+--------+----------+-------+----------+---------+------+------------+----------------+------------+----------------------+
|    1001|  Sneha Reddy|Monitor|Electronics|      12|     22000| 264000|2023-05-21|   Mumbai|  West| Meera Patel|             UPI|   Delivered|                286000|
|    1002| Ramesh Kumar|Printer|Electronics|      10|     12000| 120000|2023-08-05|    Delhi| North| Anil Sharma|     Credit Card|     Shipped|                132000|
|    1004|   Suresh Rao| Tablet|Electronics|       5|     32000| 160000|2023-01-04|    Surat|  West|  Ravi Kumar|Cash on Delivery|  Processing|                192000

In [12]:
df_silver = df_bronze \
  .dropDuplicates() \
  .dropna(subset=['order_id','product','revenue'])
df_silver = df_silver.withColumn(
    'order_date',
    to_date(col('order_date'), 'dd-MM-yyyy')
)

df_silver = df_silver \
.withColumn('order_year',year(col('order_date'))) \
.withColumn('order_month',month(col('order_date')))

df_silver = df_silver.withColumn(
    'revenue_category',
    F.when(F.col('revenue') > 40000, 'High')
    .when(F.col('revenue') > 10000, 'Medium')
    .otherwise('Low')
)

print(f'Silver layer rows: {df_silver.count()}')
print('New Columns added: order_year,order_month,revenue_category')
df_silver.select('product','revenue','order_year','order_month','revenue_category').show()

Silver layer rows: 5000
New Columns added: order_year,order_month,revenue_category
+----------+-------+----------+-----------+----------------+
|   product|revenue|order_year|order_month|revenue_category|
+----------+-------+----------+-----------+----------------+
|    Tablet| 416000|      2023|          7|            High|
|   Monitor|  88000|      2023|          7|            High|
|    Webcam|  35000|      2023|         11|          Medium|
|    Webcam|  10000|      2023|          6|             Low|
|  Keyboard|   3600|      2023|          5|             Low|
|    Tablet|  32000|      2023|         12|          Medium|
|    Tablet| 384000|      2023|          9|            High|
|    Laptop| 630000|      2023|          9|            High|
|   Speaker|  45000|      2023|          4|            High|
|  Keyboard|   7200|      2023|          3|             Low|
|Headphones|  38500|      2023|          3|          Medium|
|   Printer| 108000|      2023|         12|            High|
|H

In [13]:
df_silver = df_bronze \
  .dropDuplicates() \
  .dropna(subset=['order_id','product','revenue'])
df_silver = df_silver.withColumn(
    'order_date',
    to_date(col('order_date'), 'dd-MM-yyyy')
)

df_silver = df_silver \
.withColumn('order_year',year(col('order_date'))) \
.withColumn('order_month',month(col('order_date')))

df_silver = df_silver.withColumn(
    'revenue_category',
    F.when(F.col('revenue') > 40000, 'High')
    .when(F.col('revenue') > 10000, 'Medium')
    .otherwise('Low')
)

bronze_rows_count = df_bronze.count()
silver_rows_count = df_silver.count()
dropped_rows = bronze_rows_count - silver_rows_count

print(f'Initial rows in Bronze layer: {bronze_rows_count}')
print(f'Silver layer rows: {silver_rows_count}')
print(f'Number of items (rows) dropped: {dropped_rows}')
print('New Columns added: order_year,order_month,revenue_category')
df_silver.select('product','revenue','order_year','order_month','revenue_category').show(8)

print('\nProduct names and number of orders:')
df_silver.groupBy('product').agg(F.countDistinct('order_id').alias('number_of_orders')).show()

Initial rows in Bronze layer: 5000
Silver layer rows: 5000
Number of items (rows) dropped: 0
New Columns added: order_year,order_month,revenue_category
+--------+-------+----------+-----------+----------------+
| product|revenue|order_year|order_month|revenue_category|
+--------+-------+----------+-----------+----------------+
|  Tablet| 416000|      2023|          7|            High|
| Monitor|  88000|      2023|          7|            High|
|  Webcam|  35000|      2023|         11|          Medium|
|  Webcam|  10000|      2023|          6|             Low|
|Keyboard|   3600|      2023|          5|             Low|
|  Tablet|  32000|      2023|         12|          Medium|
|  Tablet| 384000|      2023|          9|            High|
|  Laptop| 630000|      2023|          9|            High|
+--------+-------+----------+-----------+----------------+
only showing top 8 rows

Product names and number of orders:
+----------+----------------+
|   product|number_of_orders|
+----------+-------

In [14]:
df_silver.write \
  .mode('overwrite') \
  .parquet('sales_silver.parquet')

print('Silver Parquet saved:sales_silver.parquet')
print(f'Silver size:{get_dir_size("sales_silver.parquet"):.1f} KB')

df_verify = spark.read.parquet('sales_silver.parquet')
print(f'Read-back rows:{df_verify.count()} (should match Silver count)')
df_verify.printSchema()

Silver Parquet saved:sales_silver.parquet
Silver size:65.0 KB
Read-back rows:5000 (should match Silver count)
root
 |-- order_id: integer (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- revenue: integer (nullable = true)
 |-- order_date: date (nullable = true)
 |-- city: string (nullable = true)
 |-- region: string (nullable = true)
 |-- sales_rep: string (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- order_status: string (nullable = true)
 |-- unit_price_revenue_sum: integer (nullable = true)
 |-- order_year: integer (nullable = true)
 |-- order_month: integer (nullable = true)
 |-- revenue_category: string (nullable = true)



In [16]:
top_products = df_silver \
  .groupBy('product') \
  .agg(
    F.sum('revenue').alias('total_revenue'),
    F.count('order_id').alias('num_orders'),
    F.avg('revenue').alias('avg_order_revenue')
  ) \
  .orderBy('total_revenue', ascending=False) \
  .limit(5)
print('===Top 5 Products by revenue===')
top_products.show(truncate = False)

===Top 5 Products by revenue===
+-------+-------------+----------+------------------+
|product|total_revenue|num_orders|avg_order_revenue |
+-------+-------------+----------+------------------+
|Laptop |182700000    |502       |363944.22310756973|
|Tablet |135104000    |532       |253954.8872180451 |
|Monitor|82126000     |481       |170740.12474012474|
|Printer|44544000     |488       |91278.68852459016 |
|Speaker|16317000     |470       |34717.02127659575 |
+-------+-------------+----------+------------------+



In [17]:
# query 1 : top 5 products by total revenue

top_products=df_silver \
  .groupby('product') \
  .agg(\
      F.sum('revenue').alias('total_revenue'),\
      F.count('order_id').alias('num_order'),\
      spark_round(F.avg('revenue'), 2).alias('avg_revenue')\
  ) \
  .orderBy(F.col('total_revenue').asc()) \
  .limit(10)
top_products.show(truncate=False)

+----------+-------------+---------+-----------+
|product   |total_revenue|num_order|avg_revenue|
+----------+-------------+---------+-----------+
|USB Hub   |2447400      |527      |4644.02    |
|Mouse     |3207200      |492      |6518.7     |
|Keyboard  |4878000      |495      |9854.55    |
|Webcam    |10982500     |532      |20643.8    |
|Headphones|13541500     |481      |28152.81   |
|Speaker   |16317000     |470      |34717.02   |
|Printer   |44544000     |488      |91278.69   |
|Monitor   |82126000     |481      |170740.12  |
|Tablet    |135104000    |532      |253954.89  |
|Laptop    |182700000    |502      |363944.22  |
+----------+-------------+---------+-----------+



In [18]:
revenue_by_region = df_silver \
  .groupBy('region') \
  .agg(
    F.sum('revenue').alias('total_revenue'),
    F.count('order_id').alias('total_orders'),
    F.countDistinct('customer_name').alias('unique_customers')
  ) \
  .orderBy('total_revenue', ascending=False)
print('===Revenue by Region===')
revenue_by_region.show(truncate = False)

===Revenue by Region===
+------+-------------+------------+----------------+
|region|total_revenue|total_orders|unique_customers|
+------+-------------+------------+----------------+
|West  |198275600    |2021        |15              |
|South |147145900    |1483        |15              |
|North |99878400     |995         |15              |
|East  |50547700     |501         |15              |
+------+-------------+------------+----------------+



In [19]:
monthly_revenue_trend = df_silver \
  .groupBy('order_month') \
  .agg(
    F.sum('revenue').alias('monthly_revenue'),
    F.count('order_id').alias('monthly_orders')
  ) \
  .orderBy('order_month')
print('===Monthly Revenue Trend===')
monthly_revenue_trend.show(truncate = False)

===Monthly Revenue Trend===
+-----------+---------------+--------------+
|order_month|monthly_revenue|monthly_orders|
+-----------+---------------+--------------+
|1          |41068200       |423           |
|2          |34485400       |375           |
|3          |40031200       |451           |
|4          |38857100       |390           |
|5          |39984500       |423           |
|6          |40707400       |390           |
|7          |42640700       |405           |
|8          |43718500       |418           |
|9          |37640200       |398           |
|10         |47839000       |479           |
|11         |44577100       |419           |
|12         |44298300       |429           |
+-----------+---------------+--------------+



In [20]:
monthly_revenue_trend=df_silver \
  .groupby('order_month') \
  .agg(\
      F.sum('revenue').alias('monthly_revenue'),\
      F.count('order_id').alias('monthly_order'),\
  ) \
  .orderBy(F.col('order_month').asc()) \
  .limit(12)

# Add a column for month names
monthly_revenue_trend = monthly_revenue_trend.withColumn(
    'order_month_name',
    F.date_format(F.to_date(F.concat(F.lit('2023-'), F.lpad(F.col('order_month'), 2, '0'), F.lit('-01'))), 'MMM')
)

print('=== Monthly Revenue Trend ===')
monthly_revenue_trend.select('order_month_name', 'monthly_revenue', 'monthly_order').show(truncate=False)

=== Monthly Revenue Trend ===
+----------------+---------------+-------------+
|order_month_name|monthly_revenue|monthly_order|
+----------------+---------------+-------------+
|Jan             |41068200       |423          |
|Feb             |34485400       |375          |
|Mar             |40031200       |451          |
|Apr             |38857100       |390          |
|May             |39984500       |423          |
|Jun             |40707400       |390          |
|Jul             |42640700       |405          |
|Aug             |43718500       |418          |
|Sep             |37640200       |398          |
|Oct             |47839000       |479          |
|Nov             |44577100       |419          |
|Dec             |44298300       |429          |
+----------------+---------------+-------------+

